In [3]:
# Select physical GPU 1 and hide the rest
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # now "cuda:0" == physical GPU 1


In [ ]:
import pandas as pd
df = pd.read_csv("/home/g202319370/mydesktop/DATASETS/cleaned_data_sampled_2018.csv")  



In [5]:
features = df.columns.tolist()
print(features)

['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'B

In [6]:
print("Total count:", len(df))

Total count: 360000


In [7]:
unique_classes_count = df['Label'].nunique()
print("Unique classes count:", unique_classes_count)

Unique classes count: 9


In [8]:
class_sizes = df['Label'].value_counts()
print(class_sizes)

Label
0    40000
1    40000
2    40000
3    40000
4    40000
5    40000
6    40000
7    40000
8    40000
Name: count, dtype: int64


In [9]:
# Shortened mapping
label_mapping = {
    0: "Benign",
    1: "Bot",
    2: "DDOS",
    6: "FTP-BF",
    7: "Infil"
}

# Filter only the selected classes
df_filtered = df[df['Label'].isin(label_mapping.keys())].copy()

# Apply mapping
df_filtered['ClassName'] = df_filtered['Label'].map(label_mapping)

# Optional: re-encode to 0–4
class_encoder = {name: idx for idx, name in enumerate(label_mapping.values())}
df_filtered['ClassID'] = df_filtered['ClassName'].map(class_encoder)

print(df_filtered['ClassName'].value_counts())
print(class_encoder)


ClassName
Benign    40000
Bot       40000
DDOS      40000
FTP-BF    40000
Infil     40000
Name: count, dtype: int64
{'Benign': 0, 'Bot': 1, 'DDOS': 2, 'FTP-BF': 3, 'Infil': 4}


In [10]:
# === Cell 0: Toolbox & House Style (no plots) =================================
from pathlib import Path
import time, csv, io, math, json, gc, os, resource
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle, FancyArrow, Circle
from matplotlib.lines import Line2D
from sklearn.metrics import (
    confusion_matrix, f1_score, accuracy_score, roc_auc_score,
    average_precision_score, matthews_corrcoef, precision_recall_curve,
    roc_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.decomposition import PCA

SHOW_INLINE = True  # set False later if you want save-only

# ---------- Base folders (2018 run; Q=4) ----------
ROOT     = Path("OD2018_SIMULATION_RESULT_4Q")
RES_DIR  = ROOT / "results"
FIG_DIR  = ROOT / "figs"
PRED_DIR = ROOT / "preds"
for d in (RES_DIR, FIG_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

MASTER_CSV    = RES_DIR / "master_results.csv"
FIG_MANIFEST  = RES_DIR / "figure_manifest.csv"
HIST_DIR = RES_DIR / "training_history"; HIST_DIR.mkdir(parents=True, exist_ok=True)

# ---------- House style ----------
mpl.rcParams.update({
    "font.size": 12,
    "font.family": "DejaVu Sans",
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False
})

PALETTE = {
    "wire": "#444444",
    "annot": "#212121",
    "measure_edge": "#424242",
    "amp_fill": "#e3f2fd", "amp_edge": "#1976d2",
    "vqc_fill": "#fff3e0", "vqc_edge": "#ef6c00",
    "enc_fill": "#e8f5e9", "enc_edge": "#2e7d32",
    "rot_fill": "#f3e5f5", "rot_edge": "#6a1b9a",
    "cnot_line": "#0d47a1"
}
BASE_CLASS_COLORS = {
    "DDoS": "#1976d2", "FragPkt": "#ef6c00", "MITM": "#2e7d32",
    "Normal": "#c62828", "SYN": "#6a1b9a",
}
FALLBACK_COLORS = ["#1976d2","#ef6c00","#2e7d32","#c62828","#6a1b9a","#00838f","#5d4037"]

def class_color_map(class_names):
    cmap, k = {}, 0
    for name in class_names:
        cmap[name] = BASE_CLASS_COLORS.get(name, FALLBACK_COLORS[k % len(FALLBACK_COLORS)])
        if name not in BASE_CLASS_COLORS: k += 1
    return cmap

# ---------- File/figure helpers ----------
def _ensure_header(path: Path, header: list[str]):
    if not path.exists():
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(header)

def log_figure(dataset, model, qubits, seed_or_tag, plot_type, stem, png_path, svg_path):
    header = ["dataset","model","qubits","seed_or_tag","plot_type","stem","path_png","path_svg","timestamp"]
    _ensure_header(FIG_MANIFEST, header)
    with open(FIG_MANIFEST, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([dataset, model, qubits, seed_or_tag, plot_type, stem, str(png_path), str(svg_path), int(time.time())])

def savefig_both(fig: mpl.figure.Figure, out_stem: Path, meta: dict | None = None, show: bool = SHOW_INLINE):
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    png = out_stem.with_suffix(".png"); svg = out_stem.with_suffix(".svg")
    fig.tight_layout(); fig.savefig(png, dpi=600, bbox_inches="tight"); fig.savefig(svg, bbox_inches="tight")
    if show: display(fig)
    plt.close(fig)
    if meta is not None:
        log_figure(meta["dataset"], meta["model"], meta["qubits"], meta["seed_or_tag"], meta["plot_type"], out_stem.name, png, svg)

def fig_stem(dataset: str, model: str, qubits: int, tag: str, name: str) -> Path:
    return FIG_DIR / dataset / model / f"Q{qubits}" / tag / name

def preds_dir(dataset: str, model: str, qubits: int, seed: int) -> Path:
    return PRED_DIR / dataset / model / f"Q{qubits}" / f"seed_{seed}"

def cm_blue():
    b = tuple(int(PALETTE["amp_edge"].lstrip("#")[i:i+2], 16)/255 for i in (0,2,4))
    return LinearSegmentedColormap.from_list("cm_blue", [(1,1,1), b])

# ---------- Metrics ----------
def ece_top1(y_true: np.ndarray, y_proba: np.ndarray, n_bins: int = 15) -> float:
    conf = y_proba.max(axis=1); pred = y_proba.argmax(axis=1); corr = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if np.any(m): ece += m.mean() * abs(corr[m].mean() - conf[m].mean())
    return float(ece)

def macro_auprc(y_true: np.ndarray, y_proba: np.ndarray, classes: np.ndarray) -> float:
    Y = label_binarize(y_true, classes=classes)
    return float(np.mean([average_precision_score(Y[:, k], y_proba[:, k]) for k in range(Y.shape[1])]))

# ---------- Grids ----------
ROC_FPR_GRID = np.linspace(0, 1, 1001)
PR_REC_GRID  = np.linspace(0, 1, 2001)
CALIB_BINS   = np.linspace(0, 1, 16)
CONF_BINS    = np.linspace(0, 1, 31)

# ---------- Load dataset ----------
DATASET_CSV = "/home/g202319370/mydesktop/DATASETS/cleaned_data_sampled_2018.csv"
assert os.path.exists(DATASET_CSV), f"CSV not found at {DATASET_CSV}"
df = pd.read_csv(DATASET_CSV)
print("Loaded df:", df.shape, "columns:", len(df.columns))
assert 'Label' in df.columns, "Expected a 'Label' column."

# ---------- EXACT 5-class mapping (as requested) ----------
label_mapping = {
    0: "Benign",
    1: "Bot",
    2: "DDOS",
    6: "FTP-BF",
    7: "Infil"
}

# Filter only the selected classes
df_filtered = df[df['Label'].isin(label_mapping.keys())].copy()

# Apply mapping
df_filtered['ClassName'] = df_filtered['Label'].map(label_mapping)

# Re-encode to 0–4 in the given order
class_encoder = {name: idx for idx, name in enumerate(label_mapping.values())}
df_filtered['ClassID'] = df_filtered['ClassName'].map(class_encoder)

# Quick sanity
print(df_filtered['ClassName'].value_counts())
print("class_encoder:", class_encoder)


/home/g202319370/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Loaded df: (360000, 79) columns: 79
ClassName
Benign    40000
Bot       40000
DDOS      40000
FTP-BF    40000
Infil     40000
Name: count, dtype: int64
class_encoder: {'Benign': 0, 'Bot': 1, 'DDOS': 2, 'FTP-BF': 3, 'Infil': 4}


In [ ]:
# Replace the old df with the filtered + relabeled dataset
df = df_filtered.copy()

# Optional: persist to disk for reuse
CLEAN_CSV = ROOT / "cleaned_multiclass2018.csv"
df.to_csv(CLEAN_CSV, index=False)

print("Final df ready:", df.shape)
print("Saved cleaned dataset →", CLEAN_CSV)
print("\nClass distribution:")
print(df['ClassName'].value_counts())


Final df ready: (200000, 81)
Saved cleaned dataset → OD2018_SIMULATION_RESULT_4Q/cleaned_multiclass2018.csv

Class distribution:
ClassName
Benign    40000
Bot       40000
DDOS      40000
FTP-BF    40000
Infil     40000
Name: count, dtype: int64


In [ ]:
# === Cell 0: Q5  for CIC-IDS2018 =================
import os, io, csv, time, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
import pennylane as qml

# Reuse ROOT from your toolbox cell
assert 'ROOT' in globals(), "Define ROOT in your toolbox cell first."

# ---- Config (Q=5) ----
DATASET_Q5 = "CIC-IDS2018"
MODEL_Q5   = "QuantumTabNet"
Q5         = 5
L5         = 1
TARGET_DIM5 = 2**Q5            # 32
TRAIN_FRAC5, VAL_FRAC5, TEST_FRAC5 = 0.80, 0.10, 0.10
SPLIT_SEED5, PCA_SEED5, SUBSET_SEED5 = 42, 42, 42
MODEL_SEEDS_Q5 = [101, 202, 402]

# VQC light pretrain + encoding
PRETRAIN_BUDGET5 = 3000
VQC_PRE_EPOCHS5  = 5
VQC_BATCH5       = 256
ENC_BATCH5       = 4096

# ---- Unique paths for Q5 (no clash with Q4) ----
RES_DIR_Q5  = ROOT / "results_q5q"; RES_DIR_Q5.mkdir(parents=True, exist_ok=True)
PRED_DIR_Q5 = ROOT / "preds_q5q";   PRED_DIR_Q5.mkdir(parents=True, exist_ok=True)
FEAT_DIR_Q5 = ROOT / "features_q5q" / DATASET_Q5 / MODEL_Q5 / f"Q{Q5}"; FEAT_DIR_Q5.mkdir(parents=True, exist_ok=True)
VQC_DIR_Q5  = ROOT / "vqc_weights_q5q" / DATASET_Q5 / MODEL_Q5 / f"Q{Q5}"; VQC_DIR_Q5.mkdir(parents=True, exist_ok=True)
MASTER_CSV5 = RES_DIR_Q5 / "master_q5q.csv"

def preds_dir_q5(dataset: str, model: str, q: int, seed: int) -> Path:
    p = PRED_DIR_Q5 / dataset / model / f"Q{q}" / f"seed_{seed}"
    p.mkdir(parents=True, exist_ok=True)
    return p

def _ensure_header5(path: Path, header: list[str]):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=header).writeheader()

# ---- Metrics & helpers ----
from sklearn.preprocessing import label_binarize
from sklearn.metrics import average_precision_score

def macro_auprc5(y_true, proba, classes):
    yb = label_binarize(y_true, classes=classes)
    return float(average_precision_score(yb, proba, average="macro"))

def ece_top1_5(y_true, proba, n_bins: int = 15):
    conf = proba.max(axis=1); preds = proba.argmax(axis=1)
    bins = np.linspace(0.0, 1.0, n_bins+1); ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if np.any(m):
            ece += m.mean() * abs((preds[m] == y_true[m]).mean() - conf[m].mean())
    return float(ece)

def l2_normalize_rows5(A: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.maximum(np.linalg.norm(A, axis=1, keepdims=True), eps)
    return (A / n).astype(np.float64)

def balanced_subset_indices5(y_train: np.ndarray, budget: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    cls, counts = np.unique(y_train, return_counts=True)
    per = int(min(budget // len(cls), counts.min(), len(y_train)))
    return np.hstack([rng.choice(np.where(y_train == c)[0], size=per, replace=False) for c in cls])

def extract_probs_batched5(X_amp: np.ndarray, qnode_infer, batch: int = ENC_BATCH5, out_dim: int = TARGET_DIM5) -> np.ndarray:
    N = X_amp.shape[0]
    out = np.empty((N, out_dim), dtype=np.float32)
    for s in range(0, N, batch):
        xb = torch.from_numpy(X_amp[s:s+batch]).to(dtype=torch.float64).contiguous()
        with torch.no_grad():
            pb = qnode_infer(xb).to(torch.float32).cpu().numpy()
        out[s:s+len(pb)] = pb
    return out

def count_tabnet_params5(tabnet):
    n_params = sum(p.numel() for p in tabnet.network.parameters())
    size_mb = n_params * 4 / (1024**2)
    return {"params": n_params, "size_mb": size_mb}

def get_peak_mem_mb5() -> float:
    try:
        import resource as _res
        return _res.getrusage(_res.RUSAGE_SELF).ru_maxrss / 1024.0
    except Exception:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

# Device
if torch.cuda.is_available():
    torch.cuda.set_device(0)
print("Using device:", "cuda:0" if torch.cuda.is_available() else "cpu")


Using device: cuda:0


In [ ]:
# === Cell 1 (Q5):
import os, time, csv, gc
import numpy as np, pandas as pd
import torch
import pennylane as qml
from torch import nn, optim

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, matthews_corrcoef, log_loss
from sklearn.preprocessing import label_binarize
from pytorch_tabnet.tab_model import TabNetClassifier

assert 'df' in globals(), "df not found. Load a DataFrame `df` first."

# ===== Config (Q5) =====
FORCE_VQC_PRETRAIN5 = True
FORCE_REENCODE5     = True

# One-time VQC pretrain budget (balanced across classes)
PRETRAIN_BUDGET5 = 3000   # 5 classes → ~600 per class (capped by minority size)
VQC_PRE_EPOCHS5  = 15
VQC_BATCH5       = 256
ENC_BATCH5       = 4096    # big batch for fast GPU encoding

# Small TabNet sweep on seed=101; reuse best for other seeds
DO_SWEEP5 = True
SWEEP_GRID5 = [
    dict(n_d=32, n_a=32, n_steps=4, gamma=1.0,  lambda_sparse=1e-5, mask_type="entmax"),
    dict(n_d=32, n_a=32, n_steps=6, gamma=1.5,  lambda_sparse=1e-4, mask_type="entmax"),
    dict(n_d=48, n_a=48, n_steps=4, gamma=1.5,  lambda_sparse=1e-5, mask_type="sparsemax"),
    dict(n_d=48, n_a=48, n_steps=6, gamma=1.0,  lambda_sparse=1e-4, mask_type="sparsemax"),
]
MAX_EPOCHS_TBN5, PATIENCE_TBN5 = 60, 15
LR_TBN5, WD_TBN5 = 2e-3, 1e-4

# ===== Utilities =====
def set_seed5(seed: int):
    np.random.seed(seed); torch.manual_seed(seed)

def _make_pl_device5(wires: int):
    """Prefer GPU lightning; fallback to lightning.qubit then default.qubit."""
    if torch.cuda.is_available():
        try:
            return qml.device("lightning.gpu", wires=wires)
        except Exception as e:
            print(f"[WARN Q5] lightning.gpu unavailable ({e}); falling back to CPU backend.")
    for name in ("lightning.qubit", "default.qubit"):
        try:
            return qml.device(name, wires=wires)
        except Exception:
            pass
    raise RuntimeError("No PennyLane backend available (tried lightning.gpu, lightning.qubit, default.qubit).")

# ===== Splits & Preprocess =====
df5 = df.copy()
df5.drop(columns=['Src IP','Dst IP','src_ip','dst_ip'], inplace=True, errors='ignore')

# labels (ClassID preferred; else Label/ClassName)
if 'ClassID' in df5.columns:
    y_all5 = df5['ClassID'].to_numpy()
    class_names5 = (list(df5.groupby('ClassID')['ClassName'].first().sort_index().values)
                    if 'ClassName' in df5.columns else [f"C{c}" for c in sorted(np.unique(y_all5))])
else:
    target_col5 = 'ClassName' if 'ClassName' in df5.columns else 'Label'
    le5 = LabelEncoder(); y_all5 = le5.fit_transform(df5[target_col5]); class_names5 = list(le5.classes_)

Xdf5 = df5.drop(columns=['Label','ClassName','ClassID'], errors='ignore')
Xdf5 = Xdf5.replace([np.inf, -np.inf], np.nan).select_dtypes(include=[np.number])
X_all5 = Xdf5.to_numpy()

sss_test5 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_FRAC5, random_state=SPLIT_SEED5)
tr_idx_all5, te_idx5 = next(sss_test5.split(X_all5, y_all5))
X_rem5, y_rem5 = X_all5[tr_idx_all5], y_all5[tr_idx_all5]
X_te_raw5, y_te5 = X_all5[te_idx5], y_all5[te_idx5]

val_rel5 = VAL_FRAC5 / (1.0 - TEST_FRAC5)
sss_val5 = StratifiedShuffleSplit(n_splits=1, test_size=val_rel5, random_state=SPLIT_SEED5)
tr_idx5, va_idx5 = next(sss_val5.split(X_rem5, y_rem5))
X_tr_raw5, y_tr5 = X_rem5[tr_idx5], y_rem5[tr_idx5]
X_va_raw5, y_va5 = X_rem5[va_idx5], y_rem5[va_idx5]

n_train5, n_val5, n_test5 = len(y_tr5), len(y_va5), len(y_te5)
print(f"[Q5] Split sizes → Train={n_train5}, Val={n_val5}, Test={n_test5}")

# Train-only impute + scale
train_median5 = np.nanmedian(X_tr_raw5, axis=0)
X_tr_raw5 = np.where(np.isnan(X_tr_raw5), train_median5, X_tr_raw5)
X_va_raw5 = np.where(np.isnan(X_va_raw5), train_median5, X_va_raw5)
X_te_raw5 = np.where(np.isnan(X_te_raw5), train_median5, X_te_raw5)

scaler5 = StandardScaler().fit(X_tr_raw5)
X_tr_sc5 = scaler5.transform(X_tr_raw5)
X_va_sc5 = scaler5.transform(X_va_raw5)
X_te_sc5 = scaler5.transform(X_te_raw5)

# PCA → TARGET_DIM5
t_pca05 = time.time()
pca5 = PCA(n_components=min(TARGET_DIM5, X_tr_sc5.shape[1], max(1, X_tr_sc5.shape[0]-1)),
           random_state=PCA_SEED5)
X_tr_p5 = pca5.fit_transform(X_tr_sc5)
X_va_p5 = pca5.transform(X_va_sc5)
X_te_p5 = pca5.transform(X_te_sc5)
if X_tr_p5.shape[1] < TARGET_DIM5:
    pad = TARGET_DIM5 - X_tr_p5.shape[1]
    X_tr_p5 = np.hstack([X_tr_p5, np.zeros((X_tr_p5.shape[0], pad))])
    X_va_p5 = np.hstack([X_va_p5, np.zeros((X_va_p5.shape[0], pad))])
    X_te_p5 = np.hstack([X_te_p5, np.zeros((X_te_p5.shape[0], pad))])
pca_time5 = time.time() - t_pca05
print(f"[Timing Q5] PCA: {pca_time5:.2f}s")

# L2 amplitude vectors 
X_tr_amp5 = l2_normalize_rows5(X_tr_p5)
X_va_amp5 = l2_normalize_rows5(X_va_p5)
X_te_amp5 = l2_normalize_rows5(X_te_p5)

# ===== VQC pretrain once 
VQC_DIR_Q5.mkdir(parents=True, exist_ok=True)
FEAT_DIR_Q5.mkdir(parents=True, exist_ok=True)

w_file5 = VQC_DIR_Q5 / f"vqc_weights_Q{Q5}_L{L5}.npy"
vqc_pretrain_time5 = 0.0

if w_file5.exists() and not FORCE_VQC_PRETRAIN5:
    weights_frozen_np5 = np.load(w_file5)
    print(f"[Q5] Loaded frozen VQC weights → {w_file5}")
else:
    print("[Q5] Pretraining VQC once (shared across seeds)...")
    dev_train5 = _make_pl_device5(Q5)

    @qml.qnode(dev_train5, interface="torch", diff_method="parameter-shift")
    def vqc_probs_train5(vec_batch, weights):
        qml.AmplitudeEmbedding(features=vec_batch, wires=range(Q5), normalize=False, validate_norm=False)
        qml.StronglyEntanglingLayers(weights, wires=range(Q5))
        return qml.probs(wires=range(Q5))

    set_seed5(101)
    weights5 = torch.randn((L5, Q5, 3), dtype=torch.float64, requires_grad=True)
    opt5 = optim.Adam([weights5], lr=2e-2)
    sched5 = optim.lr_scheduler.CosineAnnealingLR(opt5, T_max=VQC_PRE_EPOCHS5, eta_min=5e-4)

    sub_idx5 = balanced_subset_indices5(y_tr5, PRETRAIN_BUDGET5, seed=SUBSET_SEED5)
    X_sub5 = torch.tensor(X_tr_amp5[sub_idx5], dtype=torch.float64)
    y_sub5 = torch.tensor(y_tr5[sub_idx5], dtype=torch.long)

    t_pre5 = time.time()
    for ep in range(VQC_PRE_EPOCHS5):
        perm = torch.randperm(len(X_sub5))
        X_ep, y_ep = X_sub5[perm], y_sub5[perm]
        for i in range(0, len(X_ep), VQC_BATCH5):
            xb = X_ep[i:i+VQC_BATCH5].contiguous()
            with torch.enable_grad():
                P = vqc_probs_train5(xb, weights5)
                logits = torch.log(P.clamp_min(1e-9))
                loss = nn.NLLLoss()(logits, y_ep[i:i+VQC_BATCH5])
                opt5.zero_grad(); loss.backward(); opt5.step()
        sched5.step()
    vqc_pretrain_time5 = time.time() - t_pre5
    print(f"[Timing Q5] VQC pretraining: {vqc_pretrain_time5:.2f}s")

    weights_frozen_np5 = weights5.detach().cpu().numpy()
    np.save(w_file5, weights_frozen_np5)
    print(f"[Q5] Saved frozen VQC weights → {w_file5}")

# ===== Frozen inference qnode =====
dev_inf5 = _make_pl_device5(Q5)
weights_frozen5 = torch.from_numpy(weights_frozen_np5).to(dtype=torch.float64)

@qml.qnode(dev_inf5, interface="torch", diff_method=None)
def vqc_probs_infer5(vec_batch):
    qml.AmplitudeEmbedding(features=vec_batch, wires=range(Q5), normalize=False, validate_norm=False)
    qml.StronglyEntanglingLayers(weights_frozen5, wires=range(Q5))
    return qml.probs(wires=range(Q5))

# ===== Encode all splits (cache-aware) =====
tr_q_file5 = FEAT_DIR_Q5 / "X_tr_q.npy"
va_q_file5 = FEAT_DIR_Q5 / "X_va_q.npy"
te_q_file5 = FEAT_DIR_Q5 / "X_te_q.npy"

quantum_time5 = 0.0
have_cache5 = tr_q_file5.exists() and va_q_file5.exists() and te_q_file5.exists()
if have_cache5 and not FORCE_REENCODE5:
    X_tr_q5 = np.load(tr_q_file5); X_va_q5 = np.load(va_q_file5); X_te_q5 = np.load(te_q_file5)
    print("[Q5] Loaded cached quantum features.")
else:
    print("[Q5] Encoding all splits (frozen VQC)...")
    t_enc5 = time.time()
    X_tr_q5 = extract_probs_batched5(X_tr_amp5, vqc_probs_infer5, batch=ENC_BATCH5)
    X_va_q5 = extract_probs_batched5(X_va_amp5, vqc_probs_infer5, batch=ENC_BATCH5)
    X_te_q5 = extract_probs_batched5(X_te_amp5, vqc_probs_infer5, batch=ENC_BATCH5)
    quantum_time5 = time.time() - t_enc5
    np.save(tr_q_file5, X_tr_q5); np.save(va_q_file5, X_va_q5); np.save(te_q_file5, X_te_q5)
print(f"[Timing Q5] VQC encoding (all splits): {quantum_time5:.2f}s")

# Prepare for TabNet
X_tr_q5 = X_tr_q5.astype(np.float32)
X_va_q5 = X_va_q5.astype(np.float32)
X_te_q5 = X_te_q5.astype(np.float32)

classes_tr5 = np.unique(y_tr5)
cw5 = compute_class_weight(class_weight='balanced', classes=classes_tr5, y=y_tr5)
w_tr5 = np.array([cw5[np.where(classes_tr5 == yy)[0][0]] for yy in y_tr5]).astype(np.float32)

# ===== Tiny TabNet sweep on seed=101 =====
best_cfg5 = dict(n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-4, mask_type="entmax")
best_val_f1_5, best_val_acc_5 = -1.0, -1.0

def _fit_eval_tabnet5(cfg, seed):
    tab = TabNetClassifier(
        n_d=cfg["n_d"], n_a=cfg["n_a"], n_steps=cfg["n_steps"], gamma=cfg["gamma"],
        n_independent=2, n_shared=2, momentum=0.02,
        lambda_sparse=cfg["lambda_sparse"], optimizer_params=dict(lr=LR_TBN5, weight_decay=WD_TBN5),
        mask_type=cfg["mask_type"], verbose=0, seed=seed,
        device_name="cuda" if torch.cuda.is_available() else "cpu"
    )
    tab.fit(
        X_tr_q5, y_tr5,
        eval_set=[(X_tr_q5, y_tr5), (X_va_q5, y_va5)],
        eval_name=["train","valid"], eval_metric=["logloss","accuracy"],
        max_epochs=MAX_EPOCHS_TBN5, patience=PATIENCE_TBN5,
        batch_size=4096, virtual_batch_size=512, weights=w_tr5
    )
    proba_va = tab.predict_proba(X_va_q5); ypv = proba_va.argmax(1)
    return f1_score(y_va5, ypv, average='macro'), accuracy_score(y_va5, ypv)

if DO_SWEEP5:
    print("[Q5] TabNet sweep (seed=101)...")
    for cfg in SWEEP_GRID5:
        f1v, av = _fit_eval_tabnet5(cfg, seed=101)
        print(f"  cfg={cfg} → val F1={f1v:.4f} | val Acc={av:.4f}")
        if (f1v > best_val_f1_5) or (np.isclose(f1v, best_val_f1_5) and av > best_val_acc_5):
            best_val_f1_5, best_val_acc_5, best_cfg5 = f1v, av, cfg
    print(f"[Q5] Best TabNet cfg → {best_cfg5} | val F1={best_val_f1_5:.4f}, val Acc={best_val_acc_5:.4f}")
else:
    print("[Q5] Sweep disabled; using default TabNet cfg.")

# ===== Train per seed with best TabNet cfg =====
all_classes5 = np.arange(len(class_names5))
total_training_time_all_seeds5 = 0.0

def _write_history_csv5(tabnet, csv_path, Xtr, Ytr, Xva, Yva):
    H = getattr(tabnet, "history", None)
    hist = H.history if hasattr(H, "history") else (H if isinstance(H, dict) else None)
    out = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    if hist:
        keys = {k.lower(): k for k in hist.keys()}
        def pull(names):
            for n in names:
                if n in keys: return list(hist[keys[n]])
            return None
        epochs = max([len(v) for v in hist.values()]) if len(hist) else 0
        out["epoch"] = list(range(1, epochs+1))
        def ensure_len(a, L): return [np.nan]*L if a is None else list(a)+[np.nan]*(L-len(a))
        out["train_loss"] = ensure_len(pull(["loss","train_loss"]), epochs)
        out["val_loss"]   = ensure_len(pull(["valid_logloss","valid_loss","val_logloss","val_loss"]), epochs)
        out["train_acc"]  = ensure_len(pull(["train_accuracy","train_acc"]), epochs)
        out["val_acc"]    = ensure_len(pull(["valid_accuracy","valid_acc","val_accuracy","val_acc"]), epochs)
    else:
        out["epoch"] = [1]
        with torch.no_grad():
            proba_tr = tabnet.predict_proba(Xtr); proba_va = tabnet.predict_proba(Xva)
        out["train_loss"] = [log_loss(Ytr, proba_tr, labels=np.unique(Ytr))]
        out["val_loss"]   = [log_loss(Yva, proba_va, labels=np.unique(Ytr))]
        out["train_acc"]  = [accuracy_score(Ytr, proba_tr.argmax(1))]
        out["val_acc"]    = [accuracy_score(Yva, proba_va.argmax(1))]
    pd.DataFrame(out).to_csv(csv_path, index=False)

for SEED5 in MODEL_SEEDS_Q5:
    set_seed5(SEED5)
    seed_dir5 = preds_dir_q5(DATASET_Q5, MODEL_Q5, Q5, SEED5); seed_dir5.mkdir(parents=True, exist_ok=True)

    tabnet5 = TabNetClassifier(
        n_d=best_cfg5["n_d"], n_a=best_cfg5["n_a"], n_steps=best_cfg5["n_steps"], gamma=best_cfg5["gamma"],
        n_independent=2, n_shared=2, momentum=0.02,
        lambda_sparse=best_cfg5["lambda_sparse"], optimizer_params=dict(lr=LR_TBN5, weight_decay=WD_TBN5),
        mask_type=best_cfg5["mask_type"], verbose=0, seed=SEED5,
        device_name="cuda" if torch.cuda.is_available() else "cpu"
    )

    peak_before5 = get_peak_mem_mb5()
    t_fit5 = time.time()
    tabnet5.fit(
        X_tr_q5, y_tr5,
        eval_set=[(X_tr_q5, y_tr5), (X_va_q5, y_va5)],
        eval_name=["train","valid"], eval_metric=["logloss","accuracy"],
        max_epochs=MAX_EPOCHS_TBN5, patience=PATIENCE_TBN5,
        batch_size=4096, virtual_batch_size=512, weights=w_tr5
    )
    train_time5 = time.time() - t_fit5
    total_training_time_all_seeds5 += train_time5
    peak_mb5 = max(peak_before5, get_peak_mem_mb5())
    print(f"[Timing Q5] TabNet training (seed={SEED5}): {train_time5:.2f}s")

    # Predictions
    proba_tr5 = tabnet5.predict_proba(X_tr_q5)
    proba_va5 = tabnet5.predict_proba(X_va_q5)
    proba_te5 = tabnet5.predict_proba(X_te_q5)

    def _save_preds5(path, y_true, proba, names):
        dfp = pd.DataFrame({"y_true": y_true})
        for k, cname in enumerate(names): dfp[f"p_{cname}"] = proba[:, k]
        dfp.to_csv(path, index=False)

    _save_preds5(seed_dir5/"preds_train.csv", y_tr5, proba_tr5, class_names5)
    _save_preds5(seed_dir5/"preds_val.csv",   y_va5, proba_va5, class_names5)
    _save_preds5(seed_dir5/"preds_test.csv",  y_te5, proba_te5, class_names5)

    # Inference timing
    t_inf5 = time.time(); _ = tabnet5.predict_proba(X_te_q5)
    tabnet_ms_per_sample5 = 1000.0 * (time.time() - t_inf5) / len(X_te_q5)
    print(f"[Timing Q5] Inference (test, seed={SEED5}): {(tabnet_ms_per_sample5/1000.0)*len(X_te_q5):.4f}s ({tabnet_ms_per_sample5:.4f} ms/sample)")

    # Metrics
    pred_va5 = proba_va5.argmax(1); pred_te5 = proba_te5.argmax(1)
    yb_va5 = label_binarize(y_va5, classes=np.arange(len(class_names5)))
    yb_te5 = label_binarize(y_te5, classes=np.arange(len(class_names5)))

    macro_f1_te5 = f1_score(y_te5, pred_te5, average='macro')
    acc_te5      = accuracy_score(y_te5, pred_te5)
    auroc_te5    = roc_auc_score(yb_te5, proba_te5, average='macro', multi_class='ovr')
    auprc_te5    = macro_auprc5(y_te5, proba_te5, np.arange(len(class_names5)))
    ece_te5      = ece_top1_5(y_te5, proba_te5)
    mcc_te5      = matthews_corrcoef(y_te5, pred_te5)

    macro_f1_va5 = f1_score(y_va5, pred_va5, average='macro')
    acc_va5      = accuracy_score(y_va5, pred_va5)
    auroc_va5    = roc_auc_score(yb_va5, proba_va5, average='macro', multi_class='ovr')
    auprc_va5    = macro_auprc5(y_va5, proba_va5, np.arange(len(class_names5)))
    ece_va5      = ece_top1_5(y_va5, proba_va5)
    mcc_va5      = matthews_corrcoef(y_va5, pred_va5)

    print(f"[Q={Q5} | seed={SEED5}] PCA={pca_time5:.2f}s | Encode(all)={quantum_time5:.2f}s | TabNet={train_time5:.2f}s")
    print(f" VAL  F1={macro_f1_va5:.4f} | Acc={acc_va5:.4f} | AUROC={auroc_va5:.4f} | AUPRC={auprc_va5:.4f} | MCC={mcc_va5:.4f} | ECE={ece_va5:.4f}")
    print(f" TEST F1={macro_f1_te5:.4f} | Acc={acc_te5:.4f} | AUROC={auroc_te5:.4f} | AUPRC={auprc_te5:.4f} | MCC={mcc_te5:.4f} | ECE={ece_te5:.4f}")
    print(f" TabNet size ≈ {count_tabnet_params5(tabnet5)['size_mb']:.3f} MB")

    # History CSV (optional parity with other cell)
    hist_dir5 = RES_DIR / "training_history" / DATASET_Q5 / MODEL_Q5 / f"Q{Q5}"
    hist_dir5.mkdir(parents=True, exist_ok=True)
    def _write_hist_stub():
        H = getattr(tabnet5, "history", None)
        hist = H.history if hasattr(H, "history") else (H if isinstance(H, dict) else None)
        out = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
        if hist:
            keys = {k.lower(): k for k in hist.keys()}
            def pull(keys_try):
                for k in keys_try:
                    if k in keys: return list(hist[keys[k]])
                return None
            epochs = max([len(v) for v in hist.values()]) if len(hist) else 0
            out["epoch"] = list(range(1, epochs+1))
            def ensure_len(a, L): return [np.nan]*L if a is None else list(a)+[np.nan]*(L-len(a))
            out["train_loss"] = ensure_len(pull(["loss","train_loss"]), epochs)
            out["val_loss"]   = ensure_len(pull(["valid_logloss","valid_loss","val_logloss","val_loss"]), epochs)
            out["train_acc"]  = ensure_len(pull(["train_accuracy","train_acc"]), epochs)
            out["val_acc"]    = ensure_len(pull(["valid_accuracy","valid_acc","val_accuracy","val_acc"]), epochs)
        else:
            out["epoch"] = [1]
            with torch.no_grad():
                proba_tr = tabnet5.predict_proba(X_tr_q5); proba_va = tabnet5.predict_proba(X_va_q5)
            out["train_loss"] = [log_loss(y_tr5, proba_tr, labels=np.unique(y_tr5))]
            out["val_loss"]   = [log_loss(y_va5, proba_va, labels=np.unique(y_tr5))]
            out["train_acc"]  = [accuracy_score(y_tr5, proba_tr.argmax(1))]
            out["val_acc"]    = [accuracy_score(y_va5, proba_va.argmax(1))]
        pd.DataFrame(out).to_csv(hist_dir5 / f"history_seed_{SEED5}.csv", index=False)
    _write_hist_stub()

    # Master CSV logging
    header5 = [
        "dataset","model","variant","qubits","layers","seed",
        "train_n","val_n","test_n",
        "macro_f1_test","accuracy_test","auroc_macro_test","auprc_macro_test","mcc_test","ece_test",
        "macro_f1_val","accuracy_val","auroc_macro_val","auprc_macro_val","mcc_val","ece_val",
        "pca_s","vqc_pretrain_s","vqc_encode_all_s","tabnet_train_s",
        "encode_ms_per_sample","tabnet_ms_per_sample","pipeline_ms_per_sample",
        "peak_mem_mb","tabnet_model_size_mb","total_train_time_s","preds_test_path"
    ]
    _ensure_header5(MASTER_CSV5, header5)
    with open(MASTER_CSV5, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=header5).writerow({
            "dataset": DATASET_Q5, "model": MODEL_Q5,
            "variant": f"PCA→VQC(frozen)→TabNet", "qubits": Q5, "layers": L5, "seed": SEED5,
            "train_n": int(n_train5), "val_n": int(n_val5), "test_n": int(n_test5),
            "macro_f1_test": float(macro_f1_te5), "accuracy_test": float(acc_te5),
            "auroc_macro_test": float(auroc_te5), "auprc_macro_test": float(auprc_te5),
            "mcc_test": float(mcc_te5), "ece_test": float(ece_te5),
            "macro_f1_val": float(macro_f1_va5), "accuracy_val": float(acc_va5),
            "auroc_macro_val": float(auroc_va5), "auprc_macro_val": float(auprc_va5),
            "mcc_val": float(mcc_va5), "ece_val": float(ece_va5),
            "pca_s": float(pca_time5),
            "vqc_pretrain_s": float(vqc_pretrain_time5 if (not w_file5.exists() or FORCE_VQC_PRETRAIN5) else 0.0),
            "vqc_encode_all_s": float(quantum_time5 if (not have_cache5 or FORCE_REENCODE5) else 0.0),
            "tabnet_train_s": float(train_time5),
            "encode_ms_per_sample": float(1000.0 * (pca_time5 + (quantum_time5 if (not have_cache5 or FORCE_REENCODE5) else 0.0)) / (len(X_tr_q5)+len(X_va_q5)+len(X_te_q5))),
            "tabnet_ms_per_sample": float(tabnet_ms_per_sample5),
            "pipeline_ms_per_sample": float(1000.0 * (pca_time5 + (quantum_time5 if (not have_cache5 or FORCE_REENCODE5) else 0.0)) / (len(X_tr_q5)+len(X_va_q5)+len(X_te_q5)) + tabnet_ms_per_sample5),
            "peak_mem_mb": float(peak_mb5),
            "tabnet_model_size_mb": float(count_tabnet_params5(tabnet5)["size_mb"]),
            "total_train_time_s": float(pca_time5 + (vqc_pretrain_time5 if (not w_file5.exists() or FORCE_VQC_PRETRAIN5) else 0.0) + (quantum_time5 if (not have_cache5 or FORCE_REENCODE5) else 0.0) + train_time5),
            "preds_test_path": str(seed_dir5 / "preds_test.csv"),
        })

print(f"[Timing Q5] Total training time across {len(MODEL_SEEDS_Q5)} seeds (sum of per-seed TabNet times): {total_training_time_all_seeds5:.2f}s")
gc.collect()


[Q5] Split sizes → Train=160000, Val=20000, Test=20000
[Timing Q5] PCA: 0.12s
[Q5] Pretraining VQC once (shared across seeds)...
[Timing Q5] VQC pretraining: 976.47s
[Q5] Saved frozen VQC weights → OD2018_SIMULATION_RESULT_4Q/vqc_weights_q5q/CIC-IDS2018/QuantumTabNet/Q5/vqc_weights_Q5_L1.npy
[Q5] Encoding all splits (frozen VQC)...
[Timing Q5] VQC encoding (all splits): 138.81s
[Q5] TabNet sweep (seed=101)...

Early stopping occurred at epoch 49 with best_epoch = 34 and best_valid_accuracy = 0.8926


/home/g202319370/.local/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  cfg={'n_d': 32, 'n_a': 32, 'n_steps': 4, 'gamma': 1.0, 'lambda_sparse': 1e-05, 'mask_type': 'entmax'} → val F1=0.8920 | val Acc=0.8926
Stop training because you reached max_epochs = 60 with best_epoch = 49 and best_valid_accuracy = 0.8917


/home/g202319370/.local/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  cfg={'n_d': 32, 'n_a': 32, 'n_steps': 6, 'gamma': 1.5, 'lambda_sparse': 0.0001, 'mask_type': 'entmax'} → val F1=0.8896 | val Acc=0.8917
Stop training because you reached max_epochs = 60 with best_epoch = 55 and best_valid_accuracy = 0.8878


/home/g202319370/.local/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


... (6 outputs left out)

[Timing Q5] TabNet training (seed=202): 241.88s
[Timing Q5] Inference (test, seed=202): 0.3590s (0.0180 ms/sample)
[Q=5 | seed=202] PCA=0.12s | Encode(all)=138.81s | TabNet=241.88s
 VAL  F1=0.8940 | Acc=0.8942 | AUROC=0.9832 | AUPRC=0.9351 | MCC=0.8679 | ECE=0.0074
 TEST F1=0.8973 | Acc=0.8974 | AUROC=0.9841 | AUPRC=0.9393 | MCC=0.8719 | ECE=0.0093
 TabNet size ≈ 0.397 MB
Stop training because you reached max_epochs = 60 with best_epoch = 52 and best_valid_accuracy = 0.89575


/home/g202319370/.local/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


[Timing Q5] TabNet training (seed=402): 240.53s
[Timing Q5] Inference (test, seed=402): 0.1824s (0.0091 ms/sample)
[Q=5 | seed=402] PCA=0.12s | Encode(all)=138.81s | TabNet=240.53s
 VAL  F1=0.8941 | Acc=0.8958 | AUROC=0.9836 | AUPRC=0.9370 | MCC=0.8724 | ECE=0.0037
 TEST F1=0.8968 | Acc=0.8980 | AUROC=0.9843 | AUPRC=0.9405 | MCC=0.8746 | ECE=0.0058
 TabNet size ≈ 0.397 MB
[Timing Q5] Total training time across 3 seeds (sum of per-seed TabNet times): 686.85s


0

In [14]:
# === Cell 2: Q5 Ensemble (seeds 101/202/402) with rich metrics =================
import numpy as np, pandas as pd, csv, os
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, log_loss,
    roc_auc_score, matthews_corrcoef, classification_report, average_precision_score
)
from sklearn.preprocessing import label_binarize

# ---- Collect per-seed predictions ----
seed_files5 = []
for s in MODEL_SEEDS_Q5:
    p = (PRED_DIR_Q5 / DATASET_Q5 / MODEL_Q5 / f"Q{Q5}" / f"seed_{s}" / "preds_test.csv")
    if p.exists():
        seed_files5.append((s, p))
    else:
        print(f"Warning: missing seed {s} → {p}")
assert len(seed_files5) > 0, "No seed prediction files found. Run Cell 1 first."

dfs5 = [pd.read_csv(p) for _, p in seed_files5]

# Align columns & y_true checks
prob_cols5 = [c for c in dfs5[0].columns if c.startswith("p_")]
for i, dfi in enumerate(dfs5[1:], start=1):
    assert set(dfi.columns) == set(dfs5[0].columns), "Column mismatch across seeds (Q5)."
    dfs5[i] = dfi[dfs5[0].columns]
for i in range(1, len(dfs5)):
    assert np.array_equal(dfs5[0]["y_true"].values, dfs5[i]["y_true"].values), "y_true mismatch across seeds (Q5)."

y_true5 = dfs5[0]["y_true"].to_numpy()
class_names5b = [c[len("p_"):] for c in prob_cols5]
n_classes5 = len(class_names5b)
classes5 = np.arange(n_classes5)

# ---- Soft-vote ensemble (mean probabilities) ----
probas_list5 = [df[prob_cols5].to_numpy(dtype=float) for df in dfs5]
P_ens5 = np.mean(probas_list5, axis=0)
y_pred5 = P_ens5.argmax(axis=1)

# ---- Metrics (rich set) ----
acc5 = accuracy_score(y_true5, y_pred5)
macro_f15   = f1_score(y_true5, y_pred5, average="macro")
micro_f15   = f1_score(y_true5, y_pred5, average="micro")
weighted_f15= f1_score(y_true5, y_pred5, average="weighted")

macro_prec5   = precision_score(y_true5, y_pred5, average="macro", zero_division=0)
micro_prec5   = precision_score(y_true5, y_pred5, average="micro", zero_division=0)
weighted_prec5= precision_score(y_true5, y_pred5, average="weighted", zero_division=0)

macro_rec5    = recall_score(y_true5, y_pred5, average="macro", zero_division=0)
micro_rec5    = recall_score(y_true5, y_pred5, average="micro", zero_division=0)
weighted_rec5 = recall_score(y_true5, y_pred5, average="weighted", zero_division=0)

Y_bin5 = label_binarize(y_true5, classes=classes5)
try:
    auroc_macro5 = roc_auc_score(Y_bin5, P_ens5, average="macro", multi_class="ovr")
    auroc_micro5 = roc_auc_score(Y_bin5.ravel(), P_ens5.ravel())
except Exception:
    auroc_macro5, auroc_micro5 = np.nan, np.nan

auprc_macro5 = float(np.mean([average_precision_score(Y_bin5[:, k], P_ens5[:, k]) for k in range(n_classes5)]))
ece5 = ece_top1_5(y_true5, P_ens5)
mcc5 = matthews_corrcoef(y_true5, y_pred5)
try:
    nll5 = log_loss(y_true5, P_ens5, labels=classes5)
except Exception:
    nll5 = np.nan

print(f"\n[Ensemble Q={Q5} | {DATASET_Q5} | {MODEL_Q5}]")
print(f" Acc={acc5:.4f} | F1(macro/micro/wtd)= {macro_f15:.4f}/{micro_f15:.4f}/{weighted_f15:.4f}")
print(f" Prec(macro/micro/wtd)= {macro_prec5:.4f}/{micro_prec5:.4f}/{weighted_prec5:.4f} "
      f"| Rec(macro/micro/wtd)= {macro_rec5:.4f}/{micro_rec5:.4f}/{weighted_rec5:.4f}")
print(f" AUROC(macro)={auroc_macro5:.4f} | AUROC(micro)={auroc_micro5:.4f} | AUPRC(macro)={auprc_macro5:.4f}")
print(f" MCC={mcc5:.4f} | ECE={ece5:.4f} | NLL={nll5:.4f}")

# ---- Save ensemble predictions ----
ENS_DIR5 = PRED_DIR_Q5 / DATASET_Q5 / MODEL_Q5 / f"Q{Q5}" / "ensemble"
ENS_DIR5.mkdir(parents=True, exist_ok=True)

df_out5 = pd.DataFrame({"y_true": y_true5})
for k, cname in enumerate(class_names5b):
    df_out5[f"p_{cname}"] = P_ens5[:, k]
df_out5["y_pred"] = y_pred5
ens_preds_path5 = ENS_DIR5 / "preds_test_ensemble.csv"
df_out5.to_csv(ens_preds_path5, index=False)
print(f"Saved ensemble predictions → {ens_preds_path5}")

# ---- Classification report (per-class + aggregate) ----
rep_txt5  = classification_report(y_true5, y_pred5, target_names=class_names5b, zero_division=0)
rep_dict5 = classification_report(y_true5, y_pred5, target_names=class_names5b, output_dict=True, zero_division=0)
print("\n=== Classification Report (Ensemble Q5) ===\n")
print(rep_txt5)

rep_df5 = pd.DataFrame(rep_dict5).transpose()
rep_csv5 = ENS_DIR5 / "classification_report_ensemble.csv"
rep_df5.to_csv(rep_csv5, index=False)
print(f"Saved classification report CSV → {rep_csv5}")

# ---- Append single-row summary to master ----
header5 = [
    "dataset","model","variant","qubits","seed",
    "test_n",
    "accuracy","macro_f1","micro_f1","weighted_f1",
    "macro_precision","micro_precision","weighted_precision",
    "macro_recall","micro_recall","weighted_recall",
    "auroc_macro","auroc_micro","auprc_macro",
    "mcc","ece","nll",
    "preds_test_path"
]
_ensure_header5(MASTER_CSV5, header5)
with open(MASTER_CSV5, "a", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=header5).writerow({
        "dataset": DATASET_Q5, "model": MODEL_Q5,
        "variant": f"Ensemble(seeds {','.join(str(s) for s,_ in seed_files5)})",
        "qubits": Q5, "seed": "ENSEMBLE",
        "test_n": int(len(y_true5)),
        "accuracy": float(acc5),
        "macro_f1": float(macro_f15), "micro_f1": float(micro_f15), "weighted_f1": float(weighted_f15),
        "macro_precision": float(macro_prec5), "micro_precision": float(micro_prec5), "weighted_precision": float(weighted_prec5),
        "macro_recall": float(macro_rec5), "micro_recall": float(micro_rec5), "weighted_recall": float(weighted_rec5),
        "auroc_macro": float(auroc_macro5), "auroc_micro": float(auroc_micro5), "auprc_macro": float(auprc_macro5),
        "mcc": float(mcc5), "ece": float(ece5), "nll": float(nll5),
        "preds_test_path": str(ens_preds_path5),
    })

print("Appended ensemble summary to MASTER_CSV5.")



[Ensemble Q=5 | CIC-IDS2018 | QuantumTabNet]
 Acc=0.9018 | F1(macro/micro/wtd)= 0.9009/0.9018/0.9009
 Prec(macro/micro/wtd)= 0.9055/0.9018/0.9055 | Rec(macro/micro/wtd)= 0.9018/0.9018/0.9018
 AUROC(macro)=0.9849 | AUROC(micro)=0.9941 | AUPRC(macro)=0.9426
 MCC=0.8788 | ECE=0.0116 | NLL=0.1827
Saved ensemble predictions → OD2018_SIMULATION_RESULT_4Q/preds_q5q/CIC-IDS2018/QuantumTabNet/Q5/ensemble/preds_test_ensemble.csv

=== Classification Report (Ensemble Q5) ===

              precision    recall  f1-score   support

      Benign       0.81      0.66      0.73      4000
         Bot       1.00      1.00      1.00      4000
        DDOS       1.00      1.00      1.00      4000
      FTP-BF       1.00      1.00      1.00      4000
       Infil       0.71      0.85      0.78      4000

    accuracy                           0.90     20000
   macro avg       0.91      0.90      0.90     20000
weighted avg       0.91      0.90      0.90     20000

Saved classification report CSV → OD2018_

/home/g202319370/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
